In [1]:
import pandas as pd


In [2]:
df_N_N1 = pd.read_pickle("./NEW_filtered_N_N1.pkl")

In [4]:
df_N_N1

,from_year,to_year,year_N,year_N1,HS_N,HS_N1,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,Description_N,Description_N1
0,2022,2023,2022,2023,01023100,01023100,1.000000,1.000000,1.000000,1.000000,Live purebred breeding buffalo,Live purebred breeding buffalo
1,2022,2023,2022,2023,01019040,01019040,1.000000,1.000000,1.000000,1.000000,Mules and hinnies not imported for immediate s...,Mules and hinnies not imported for immediate s...
2,2022,2023,2022,2023,01031000,01031000,1.000000,1.000000,1.000000,1.000000,Live purebred breeding swine,Live purebred breeding swine
3,2022,2023,2022,2023,01022100,01022100,1.000000,1.000000,1.000000,1.000000,Live purebred breeding cattle,Live purebred breeding cattle
4,2022,2023,2022,2023,01012900,01012900,1.000000,1.000000,1.000000,1.000000,Live horses other than purebred breeding horses,Live horses other than purebred breeding horses
...,...,...,...,...,...,...,...,...,...,...,...,...
745501,1789,1790,1789,1790,1789_56_canes_walking_sticks_and_whips,1790_129_canes_walking_sticks_and_whips,1.000000,1.000000,1.000000,1.000000,"canes, walking sticks and whips","canes, walking sticks and whips"
745502,1789,1790,1789,1790,1789_61_playing_cards,1790_110_playing_cards,1.000000,1.000000,1.000000,1.000000,playing cards,playing cards
745503,1789,1790,1789,1790,1789_62_every_coach_chariot_or_other_four_wheel,1790_138_all_coaches_chariots_phaetons_chaises_ch,0.836723,0.120000,0.594340,0.669140,"every coach, chariot or other four wheel carri...","All coaches, chariots, phaetons, chaises, chai..."
745504,1789,1790,1789,1790,1789_58_brushes,1790_131_brushes,1.000000,1.000000,1.000000,1.000000,brushes,brushes


In [12]:
import pandas as pd
from collections import defaultdict, deque, Counter

# ---------------------------------------------------------------
# 0) Load your mapping table (N  →  N+1)
# ---------------------------------------------------------------
df = pd.read_pickle(
    "./NEW_filtered_N_N1.pkl"
)
# normalise description strings once
df['desc_N']  = df['Description_N'] .str.strip().str.lower()
df['desc_N1'] = df['Description_N1'].str.strip().str.lower()

# ensure the destination columns exist
for col in ['Mapped_2023_HS','Mapped_2023_Description']:
    if col not in df.columns:
        df[col] = None

# ---------------------------------------------------------------
# 1) Build predecessor list:  (year_N1, desc_N1) -> [row_idx …]
# ---------------------------------------------------------------
pred = defaultdict(list)
for idx, r in df.iterrows():
    pred[(r.year_N1, r.desc_N1)].append(idx)

# ---------------------------------------------------------------
# 2) Seed queue with every row whose *target year* is 2023
# ---------------------------------------------------------------
queue   = deque()
visited = set()

for idx, r in df[df.year_N1 == 2023].iterrows():
    anchor_hs   = r.HS_N1
    anchor_desc = r.Description_N1

    # stamp the seed row itself
    df.at[idx, 'Mapped_2023_HS']          = anchor_hs
    df.at[idx, 'Mapped_2023_Description'] = anchor_desc

    # push predecessor node (year_N , desc_N)
    queue.append( (r.year_N, r.desc_N, anchor_hs, anchor_desc) )

# ---------------------------------------------------------------
# 3) Description-only DFS/BFS backward
# ---------------------------------------------------------------
while queue:
    yr, desc, anchor_hs, anchor_desc = queue.pop()
    node = (yr, desc)
    if node in visited:
        continue
    visited.add(node)

    for idx in pred.get(node, []):
        if df.at[idx, 'Mapped_2023_HS'] is None:
            df.at[idx, 'Mapped_2023_HS']          = anchor_hs
            df.at[idx, 'Mapped_2023_Description'] = anchor_desc

            # push the predecessor of that row
            prev = df.loc[idx]
            queue.append( (prev.year_N, prev.desc_N, anchor_hs, anchor_desc) )

# ---------------------------------------------------------------
# 4) Diagnostics
# ---------------------------------------------------------------
n_blank = df['Mapped_2023_HS'].isna().sum()
print(f"\nRows still without a 2023 mapping: {n_blank:,}")

# a) blank count per year
per_year_blank = df[df['Mapped_2023_HS'].isna()].groupby('year_N').size()
print("\nUnmapped rows by year (top 10):")
print(per_year_blank.sort_values(ascending=False).head(10))

# b) duplicate descriptions per year (rows sharing same desc_N)
dupes = (
    df.groupby(['year_N','desc_N'])
      .size()
      .loc[lambda s: s > 1]        # keep only duplicates
      .groupby(level=0)
      .size()
)
print("\nYears with ≥2 identical descriptions:")
print(dupes.sort_index().head(15))

# ---------------------------------------------------------------
# 5) Save result if desired
# df.to_pickle("filtered_N_N1_with_2023_map_desc_only.pkl")



Rows still without a 2023 mapping: 0

Unmapped rows by year (top 10):
Series([], dtype: int64)

Years with ≥2 identical descriptions:
year_N
1790      1
1816      4
1824      6
1828      3
1861     25
1883     43
1894     14
1897    119
1913      1
1922      5
1930    513
1931    407
1932    407
1933    407
1934    407
dtype: int64


In [14]:
# save df as a pickle file: 1789_2023_MAPPINGS_N_N1.pkl
df.to_pickle(
    "./1789_2023_MAPPINGS_N_N1.pkl"
)

In [15]:
df = pd.read_pickle(
    "./1789_2023_MAPPINGS_N_N1.pkl"
)

In [18]:
df

,from_year,to_year,year_N,year_N1,HS_N,HS_N1,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,Description_N,Description_N1,desc_N,desc_N1,Mapped_2023_HS,Mapped_2023_Description
0,2022,2023,2022,2023,01023100,01023100,1.000000,1.000000,1.000000,1.000000,Live purebred breeding buffalo,Live purebred breeding buffalo,live purebred breeding buffalo,live purebred breeding buffalo,01023100,Live purebred breeding buffalo
1,2022,2023,2022,2023,01019040,01019040,1.000000,1.000000,1.000000,1.000000,Mules and hinnies not imported for immediate s...,Mules and hinnies not imported for immediate s...,mules and hinnies not imported for immediate s...,mules and hinnies not imported for immediate s...,01019040,Mules and hinnies not imported for immediate s...
2,2022,2023,2022,2023,01031000,01031000,1.000000,1.000000,1.000000,1.000000,Live purebred breeding swine,Live purebred breeding swine,live purebred breeding swine,live purebred breeding swine,01031000,Live purebred breeding swine
3,2022,2023,2022,2023,01022100,01022100,1.000000,1.000000,1.000000,1.000000,Live purebred breeding cattle,Live purebred breeding cattle,live purebred breeding cattle,live purebred breeding cattle,01022100,Live purebred breeding cattle
4,2022,2023,2022,2023,01012900,01012900,1.000000,1.000000,1.000000,1.000000,Live horses other than purebred breeding horses,Live horses other than purebred breeding horses,live horses other than purebred breeding horses,live horses other than purebred breeding horses,01012900,Live horses other than purebred breeding horses
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745501,1789,1790,1789,1790,1789_56_canes_walking_sticks_and_whips,1790_129_canes_walking_sticks_and_whips,1.000000,1.000000,1.000000,1.000000,"canes, walking sticks and whips","canes, walking sticks and whips","canes, walking sticks and whips","canes, walking sticks and whips",32041940,Synthetic organic coloring matter and preparat...
745502,1789,1790,1789,1790,1789_61_playing_cards,1790_110_playing_cards,1.000000,1.000000,1.000000,1.000000,playing cards,playing cards,playing cards,playing cards,20089929,"Grapes, otherwise prepared or preserved, nesoi"
745503,1789,1790,1789,1790,1789_62_every_coach_chariot_or_other_four_wheel,1790_138_all_coaches_chariots_phaetons_chaises_ch,0.836723,0.120000,0.594340,0.669140,"every coach, chariot or other four wheel carri...","All coaches, chariots, phaetons, chaises, chai...","every coach, chariot or other four wheel carri...","all coaches, chariots, phaetons, chaises, chai...",32041940,Synthetic organic coloring matter and preparat...
745504,1789,1790,1789,1790,1789_58_brushes,1790_131_brushes,1.000000,1.000000,1.000000,1.000000,brushes,brushes,brushes,brushes,63025930,"Table linen, of textile materials other than o..."
